# Workflow or agent?

Everything so far had us in control: we wrote the prompt, we decided what
went into the context, we chose when to call the model. That is a
**workflow** - we write the control flow, the model fills in the steps.

An **agent** changes one thing: the model decides the control flow, it chooses
what to do next, and it keeps going until it decides it is done.

That single change buys flexibility but removes predictability and testability with the addition of higher costs. The notebook builds both on the same task so we can see the trade.

## Part 0 - Setup

Run this cell first, in every notebook. It fetches the course repository into
the Colab session and moves into the `notebooks/` folder, so that the
`../data/...` paths work.

It is safe to run more than once, and safe after a restart.

Note: outside Colab the cell does nothing except report the working directory.
Start Jupyter from inside `notebooks/` and the paths work the same way.

If it prints `data ok: True`, we are set.

In [1]:
# --- SETUP: run this first ---
# works in Colab and locally, safe to run more than once
import os, sys, subprocess
REPO = "ai_bootcamp_foundations"
if "google.colab" in sys.modules:
    if not os.path.isdir(f"/content/{REPO}"):
        subprocess.run(["git", "clone", "-q",
                        f"https://github.com/mgrubisic/{REPO}.git"],
                       cwd="/content", check=True)
    os.chdir(f"/content/{REPO}/notebooks")
print("cwd:", os.getcwd(), "| data ok:", os.path.isdir("../data"))

cwd: /content/ai_bootcamp_foundations/notebooks | data ok: True


`task="agent"` picks the chain for models that support tool calling. An agent
loop makes 5-15 calls per run, so anything that goes wrong continues in the wrong direction several
times over.

In [2]:
import json, time, io, contextlib, traceback, re

API_KEY = None
if "google.colab" in sys.modules:
    from google.colab import userdata
    try:
        API_KEY = userdata.get("GOOGLE_API_KEY")
    except Exception as e:
        print("secret not available:", type(e).__name__)
else:
    API_KEY = os.environ.get("GOOGLE_API_KEY")

TASK = "agent"

from lares_llm import (set_key, ask, call, usage,
                       CHAINS, STATS, LAST)
set_key(API_KEY)

print("key:", "OK" if API_KEY else "MISSING - see notebook 4b, Part 0")
print(f"chain for '{TASK}': {' > '.join(CHAINS[TASK])}")

key: OK
chain for 'agent': gemma-4-31b-it > gemini-3.1-flash-lite > gemini-3.5-flash-lite


## Part 1: The task

Same job for both frameworks, on the Ames housing dataset we already discussed:

> Load the house price data, find the three features most correlated with the
> sale price, check them for missing values, and report the result.

Chosen because: it has a **known correct answer**, so we can score both
approaches.

In [3]:
DATA = "../data/housing_prices/housing.csv"
print("data file:", DATA, "|", "found" if os.path.exists(DATA) else "NOT FOUND")
if not os.path.exists(DATA):
    print("\navailable csv files:")
    import glob
    for f in sorted(glob.glob("../data/**/*.csv", recursive=True))[:10]:
        print("  ", f)

TASK_TEXT = ("Load the house price data, find the three numeric features most "
             "correlated with SalePrice, and check those three for missing "
             "values. Report the result.")

data file: ../data/housing_prices/housing.csv | found


## Part 2: The workflow

Three fixed steps. **We** wrote the order; the model is in one of
them.

Notice what this enables:
- the number of LLM calls is known in advance,
- the code that works with the data is ours,
- and the whole process (or its parts) can be unit tested.

Notice also the inherent cost: if the question changes, we have to rewrite the pipeline.

In [4]:
import pandas as pd, numpy as np

def workflow(path: str) -> dict:

    """fixed pipeline: we code the work, the model writes the summary"""
    t0 = time.time()
    calls_before = STATS["requests"]

    # step 1 - our code, deterministic, calculate correlation of all variables, find top 3
    df = pd.read_csv(path)
    num = df.select_dtypes(include=np.number).drop(columns=["SalePrice"], errors="ignore")
    corr = num.corrwith(df["SalePrice"]).abs().sort_values(ascending=False)
    top3 = corr.head(3)

    # step 2 - our code, deterministic, calculated missing data for top 3 most correlated variables
    missing = df[list(top3.index)].isna().sum().to_dict()

    # step 3 - the LLM qeuiry: turning precalculated numbers into a sentence
    facts = {"top3": {k: round(float(v), 3) for k, v in top3.items()},
             "missing": missing, "rows": len(df)}
    summary = ask(f"Write two sentences summarising these findings for an "
                  f"engineer. Use the numbers as given.\n\n{json.dumps(facts)}",
                  task=TASK, max_tokens=300, verbose=False)

    return {"facts": facts, "summary": summary,
            "llm_calls": STATS["requests"] - calls_before,
            "seconds": time.time() - t0}


wf = workflow(DATA)
print(json.dumps(wf["facts"], indent=1))
print("\nsummary:", wf["summary"])
print(f"\nLLM calls: {wf['llm_calls']}  |  {wf['seconds']:.1f}s")

{
 "top3": {
  "OverallQual": 0.791,
  "GrLivArea": 0.709,
  "GarageCars": 0.64
 },
 "missing": {
  "OverallQual": 0,
  "GrLivArea": 0,
  "GarageCars": 0
 },
 "rows": 1460
}

summary: The dataset contains 1460 rows with no missing values for the primary features. The top three predictors are OverallQual (0.791), GrLivArea (0.709), and GarageCars (0.64).

LLM calls: 1  |  107.6s


## Part 3: The agent

Same job, but the model decides what to do and how to do it. We provide one tool and a loop.

Three pieces of code:

- **a tool** - a name, a description, and a JSON schema for its arguments
- **a sandbox** - our code that actually executes what the model asks for
- **a loop** - call, execute, feed the result back, call again

Look at the schema. It is the structured output from notebook 4b, Part 4, under
a different name. A tool call *is* structured output.

In [5]:
TOOL = {
    "name": "run_python",
    "description": ("Run Python code. State persists between calls, so variables "
                    "stay. Use print() to see anything - only stdout is returned. "
                    "pandas is available as pd, numpy as np. No internet access."),
    "parameters": {
        "type": "object",
        "properties": {"code": {"type": "string", "description": "Python code"}},
        "required": ["code"],
    },
}


class Sandbox:
    """persistent namespace (self.ns), with output truncation"""

    def __init__(self, max_chars: int = 3000):
        self.ns = {"pd": pd, "np": np}
        self.max_chars = max_chars
        self.calls = 0

    def __call__(self, code: str) -> str:
        self.calls += 1
        buf = io.StringIO()
        try:
            with contextlib.redirect_stdout(buf), contextlib.redirect_stderr(buf):
                exec(code, self.ns)
        except Exception:
            # the traceback goes BACK to the model - that is how it self-corrects
            buf.write("\n--- TRACEBACK ---\n" + traceback.format_exc(limit=3))

        out = buf.getvalue() or "(no output - use print() to see something)"
        if len(out) > self.max_chars:
            # one print(df) can otherwise blow the whole context window
            head, tail = out[:self.max_chars // 2], out[-self.max_chars // 2:]
            out = f"{head}\n... [{len(out) - self.max_chars} chars cut] ...\n{tail}"
        return out

In [7]:
AGENT_SYSTEM = ("You are a data analyst. You have one tool, run_python. "
                "Work step by step, checking each result before the next step. "
                "When you are finished, reply with plain text and NO tool call.")


def agent(task_text: str, max_iter: int = 12, verbose: bool = False) -> dict:
    """the loop: call, execute, feed back, repeat"""
    sb = Sandbox()
    contents = [{"role": "user", "parts": [{"text": task_text}]}]
    t0, calls_before = time.time(), STATS["requests"]
    diag = {"turns": 0, "tool_calls": 0, "malformed": 0, "stopped": "max_iter",
            "answer": "", "peak_input": 0}

    for turn in range(1, max_iter + 1):
        diag["turns"] = turn
        res = call(contents, tools=[TOOL], task=TASK, system=AGENT_SYSTEM,
                   max_tokens=2048, verbose=verbose)
        diag["peak_input"] = max(diag["peak_input"], LAST["input"])

        if res["finish"] == "ERROR":
            diag["stopped"] = "error"
            diag["answer"] = res["text"]
            break

        # no tool call -> the model considers itself done
        if not res["tool_calls"]:
            diag["stopped"] = "done"
            diag["answer"] = res["text"]
            if verbose:
                print(f"  [{turn}] finished")
            break

        # rebuild the model turn so the next call sees what it just did
        parts = ([{"text": res["text"]}] if res["text"] else []) \
            + [{"functionCall": fc} for fc in res["tool_calls"]]
        contents.append({"role": "model", "parts": parts})

        for tc in res["tool_calls"]:
            diag["tool_calls"] += 1
            code = (tc.get("args") or {}).get("code")
            if not isinstance(code, str) or not code.strip():
                diag["malformed"] += 1
                out = "ERROR: missing string argument 'code'."
                if verbose:
                    print(f"  [{turn}] MALFORMED tool call")
            else:
                out = sb(code)
                if verbose:
                    print(f"  [{turn}] run_python ({len(code)} chars) -> "
                          f"{out.strip()!r}")
            contents.append({"role": "user", "parts": [{"functionResponse": {
                "name": tc["name"], "response": {"output": out}}}]})

    # ran out of turns -> ask once more with NO tools, so it must answer in text
    if diag["stopped"] == "max_iter":
        contents.append({"role": "user", "parts": [{"text":
            "You have run out of turns. Summarise what you found so far, "
            "with your best answer given the evidence. No tool call."}]})
        res = call(contents, task=TASK, system=AGENT_SYSTEM,
                   max_tokens=2048, verbose=verbose)
        diag["answer"] = res["text"]

    diag["llm_calls"] = STATS["requests"] - calls_before
    diag["seconds"] = time.time() - t0
    diag["sandbox_calls"] = sb.calls
    return diag


ag = agent(f"{TASK_TEXT} The file is at {DATA!r}.")
print(f"\n[{ag['stopped']}] answer:", ag["answer"])


[error] answer: [all models failed]


## Part 4: The comparison

Compare the workflow vs. the agentic approach.

In [8]:
print(f"{'':22s} {'workflow':>12s} {'agent':>12s}")
print("-" * 50)
rows = [("LLM calls", wf["llm_calls"], ag["llm_calls"]),
        ("seconds", f"{wf['seconds']:.1f}", f"{ag['seconds']:.1f}"),
        ("our code", "all", "the loop"),
        ("model code", "none", f"{ag['sandbox_calls']} blocks"),
        ("steps known upfront", "yes", "no"),
        ("unit testable", "yes", "not really"),
        ("works on a new question", "no", "yes")]
for label, a, b in rows:
    print(f"{label:22s} {str(a):>12s} {str(b):>12s}")

print(f"\nagent finished on its own: {ag['stopped'] == 'done'}")
print(f"malformed tool calls: {ag['malformed']}")
usage("last call")

                           workflow        agent
--------------------------------------------------
LLM calls                         1            2
seconds                       107.6         88.1
our code                        all     the loop
model code                     none     2 blocks
steps known upfront             yes           no
unit testable                   yes   not really
works on a new question           no          yes

agent finished on its own: False
malformed tool calls: 0
  last call: gemma-4-31b-it  in 1399  out 161  thinking 0  30.4s
  total: 4 calls  in 1829  out 316  thinking 0


The agent cost several times more calls and time to reach the same answer,
and the workflow cannot be wrong about the correlations because *our* code
computed them.

On this task the workflow wins on every property that matters.

## Part 5: The task where the workflow breaks

Change the question:

> Find something unusual in this dataset.

Now let's write the pipeline. Can we? We can't but not because it is hard, because we
do not know which steps to write. "Unusual" is not a computation until you
have looked.

An agent is worth its cost when **we cannot enumerate
the steps in advance**.

In [9]:
OPEN_TASK = (f"Look at the dataset at {DATA!r} and find something unusual or "
             "surprising about it. Investigate whatever you think is worth "
             "checking, then tell me what you found and why it matters.")

ag2 = agent(OPEN_TASK, max_iter=12)
print("\n=== what it found ===")
print(ag2["answer"])
print(f"\nturns: {ag2['turns']}  |  tool calls: {ag2['tool_calls']}  |  "
      f"{ag2['seconds']:.1f}s  |  peak input: {ag2['peak_input']:,} tokens")


=== what it found ===
After analyzing the dataset, I found a highly unusual entry that stands out as a significant outlier.

**The Finding:**
There is a property (index 1298) that has the **largest living area in the entire dataset (5,642 sq ft)** and a perfect **Overall Quality score of 10**, yet it sold for only **$160,000**.

To put this in perspective:
*   The average sale price in the dataset is approximately **$180,921**.
*   Other houses with similar living areas (e.g., index 1182 and 691) sold for **$745,000** and **$755,000** respectively.
*   This house is nearly 3.5 times larger than the median home, but it sold for less than the average home.

**Why it matters:**
This is a classic "extreme outlier." In a real estate context, such a discrepancy usually suggests one of three things:
1.  **Data Entry Error:** The square footage or the sale price was typed incorrectly.
2.  **Distressed Sale:** The property may have been sold under extreme circumstances (e.g., a family transfer

Check:

1. **Is the finding real?** We check ourselves.
2. **Would we have found it?** If yes, the agent saved us time, but if not, it did
   something beyond a workflow could not.

### The guardrails


- **`max_iter`** - a model that never stops will loop until the quota is gone.
On hitting the cap, one last call **without the tool** forces a text answer;
`stopped` marks it as best-effort. Try setting it to 2.

- **Output truncation in the sandbox** - one `print(df)` on a wide table sends
tens of thousands of tokens back into the context. That is the fastest way to
hit a token-per-minute limit, and it looks like the model being slow.

- **Tracebacks go back to the model** - this is a guardrail *for* the agent, not
against it. Handing back the error is what lets it self-correct, and it is why
the loop needs no retry logic of its own.

- **One tool, and it has no side effects** - `run_python` reads files and prints.
It sends no email, writes no database. In a question-answering system the worst
case is a wrong answer; **in an agent with tools the worst case is an
action.**

In [10]:
# deliberately break it: no room to finish
short = agent(f"{TASK_TEXT} The file is at {DATA!r}.", max_iter=1, verbose=False)
print(f"max_iter=1  -> stopped: {short['stopped']}, turns: {short['turns']}")
print(f"             answer: {short['answer']!r}")

# and what an untruncated tool result would have cost
sb_test = Sandbox(max_chars=3000)
big = sb_test(f"import pandas as pd; print(pd.read_csv({DATA!r}).head(20).to_string())")
raw_len = len(pd.read_csv(DATA).head(20).to_string())
print(f"\nuntruncated tool output: {raw_len:,} chars (~{raw_len//3.7:,.0f} tokens)")
print(f"after truncation:        {len(big):,} chars")

max_iter=1  -> stopped: max_iter, turns: 1
             answer: 'I have loaded the house price dataset and confirmed it contains 1,460 entries and 81 columns, including the target variable `SalePrice`. \n\nWhile I have not yet executed the correlation analysis or the missing value check for the specific top three features, the initial data inspection shows that the dataset contains a mix of numeric (int64, float64) and categorical (object) features. \n\nTo complete the request, the next steps would be to calculate the correlation matrix for all numeric columns, identify the three features with the highest absolute correlation to `SalePrice`, and then run `.isnull().sum()` on those specific columns.'

untruncated tool output: 18,836 chars (~5,090 tokens)
after truncation:        3,027 chars


## HANDS-ON: log three ways it broke

Run the agent a few more times - change the task, lower `max_iter`, ask for
something the data cannot answer, ask in Croatian. **Write down three distinct
ways it failed**, and for each one:

- what you saw
- what caused it
- what would have caught it in production (guardrails?)

Failure modes worth expecting: it never emits a final answer; it invents a
column name; it "verifies" something without running any code; it reports a
number that its own tool output contradicts; it succeeds but takes nine turns
to do three turns of work.

In [ ]:
# space for your notes
failures = [
    # {"saw": "...", "cause": "...", "guardrail": "..."},
]

for i, f in enumerate(failures, 1):
    print(f"{i}. {f['saw']}\n   cause: {f['cause']}\n   guardrail: {f['guardrail']}")
print(f"\n{len(failures)} logged")


0 logged
